In [1]:
#%pip install pandas h5py numpy scipy matplotlib torch torchvision torchaudio -i https://pypi.tuna.tsinghua.edu.cn/simple --break-system-packages
#!pip install mat73
# %pip install pandas -i https://pypi.org/simple/
# %pip install h5py numpy scipy matplotlib -i https://pypi.org/simple/
# !nvidia-smi
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import scipy 


In [3]:
w1, w2 = 35, 35               # Network input block size
s1, s2 = 4, 4                 # Stride for original data (Amir/coh)
SWAN_STRIDE = 35              # SWAN block stride (non-overlapping)
SAMPLE_RATIO_SWAN = 1.0       # Sampling ratio per SWAN category
BATCH_SIZE = 512
TRAIN_RATIO = 1.0

In [4]:
s1=4
s2=4
def patch2d(A, l1=35, l2=35, s1=s1, s2=s2, mode=1):
    [n1, n2] = A.shape
    if mode == 1:
        tmp = np.mod(n1 - l1, s1)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([s1 - tmp, n2])), axis=0)
        tmp = np.mod(n2 - l2, s2)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([A.shape[0], s2 - tmp])), axis=1)
        [N1, N2] = A.shape
        X = []
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                tmp = np.reshape(A[i1:i1+l1, i2:i2+l2], (l1*l2, 1), order='F')
                X.append(tmp)
        X = np.array(X)
    else:
        pass
    return X[:, :, 0]

def patch2d_inv(X, n1, n2, l1=35, l2=35, s1=s1, s2=s2, mode=1):
    if mode == 1:
        tmp1 = np.mod(n1 - l1, s1)
        tmp2 = np.mod(n2 - l2, s2)
        if tmp1 != 0 and tmp2 != 0:
            A = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
            mask = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
        if tmp1 != 0 and tmp2 == 0:
            A = np.zeros([n1 + s1 - tmp1, n2])
            mask = np.zeros([n1 + s1 - tmp1, n2])
        if tmp1 == 0 and tmp2 != 0:
            A = np.zeros([n1, n2 + s2 - tmp2])
            mask = np.zeros([n1, n2 + s2 - tmp2])
        if tmp1 == 0 and tmp2 == 0:
            A = np.zeros([n1, n2])
            mask = np.zeros([n1, n2])
        [N1, N2] = A.shape
        id = -1
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                id = id + 1
                A[i1:i1+l1, i2:i2+l2] = A[i1:i1+l1, i2:i2+l2] + np.reshape(X[id, :], [l1, l2], order='F')
                mask[i1:i1+l1, i2:i2+l2] = mask[i1:i1+l1, i2:i2+l2] + np.ones([l1, l2])
        A = A / mask
        A = A[0:n1, 0:n2]
    else:
        pass
    return A


In [5]:
def load_swan_category(npz_path, sample_ratio=0.10, noise_sigma=0.0):
   
    data = np.load(npz_path)
    patches = data['patches']
    N = patches.shape[0]
    
    np.random.seed(42)
    idx = np.random.choice(N, int(N * sample_ratio), replace=False)
    selected = patches[idx]  # (M, 128, 128)
    M = selected.shape[0]
    
    blocks = []
    block_size = 35
    
    for i in range(0, 128 - block_size + 1, SWAN_STRIDE):
        for j in range(0, 128 - block_size + 1, SWAN_STRIDE):
            block = selected[:, i:i+block_size, j:j+block_size].reshape(M, -1)
            blocks.append(block)
    clean_vecs = np.concatenate(blocks, axis=0).astype(np.float32)  # (total_blocks, 1225)
    
    
    noisy_vecs = clean_vecs.copy()
    
    return noisy_vecs, clean_vecs



import h5py
import numpy as np

def load_mat73(file_path):

    
    data_dict = {}
    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            data_dict[key] = np.array(f[key]).T
    return data_dict

In [6]:

print("Loading AmirSyn2d and coh2d ...")
data0 = sio.loadmat('AmirSyn2d.mat')
data1 = load_mat73('coh2d.mat')

X_noisy0 = patch2d(data0['Noisy'], w1, w2, s1, s2)
X_clean0 = patch2d(data0['Data'],  w1, w2, s1, s2)

X_noisy1 = patch2d(data1['Noisy'], w1, w2, s1, s2)
X_clean1 = X_noisy1.copy()

print(f"Amir: noisy {X_noisy0.shape}, clean {X_clean0.shape}")
print(f"coh:  noisy {X_noisy1.shape}, no clean data (用 noisy 顶替)")

# ==================== Step 2: Load four major SWAN categories (10% each). ====================
swan_files = [
    'SWAN_real_poststack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz',
    'SWAN_real_prestack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz',
    'SWAN_syn_poststack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz',
    'SWAN_syn_prestack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz'
]

swan_noisy_list, swan_clean_list = [], []
for f in swan_files:
    try:
        n, c = load_swan_category(f, SAMPLE_RATIO_SWAN, 0.0)
        swan_noisy_list.append(n)
        swan_clean_list.append(c)
        print(f"{f}: {n.shape[0]} blocks loaded")
    except FileNotFoundError:
        print(f"Warning: {f} not found, skipping...")

if swan_noisy_list:
    X_noisy_swan = np.concatenate(swan_noisy_list, axis=0)
    X_clean_swan = np.concatenate(swan_clean_list, axis=0)
else:
    X_noisy_swan = np.empty((0, w1*w2), dtype=np.float32)
    X_clean_swan = np.empty((0, w1*w2), dtype=np.float32)

# ==================== Step 3: Merge all data. ====================
X_noisy = np.concatenate([X_noisy0, X_noisy1, X_noisy_swan], axis=0)
X_clean = np.concatenate([X_clean0, X_clean1, X_clean_swan], axis=0)

print(f"\nTotal noisy shape: {X_noisy.shape}")
print(f"Total clean shape (for evaluation only): {X_clean.shape}")

# ==================== Step 4: Convert to Tensor, randomly shuffle ====================
X_noisy = torch.from_numpy(X_noisy).float()
X_clean = torch.from_numpy(X_clean).float()

N_total = X_noisy.size(0)
perm = torch.randperm(N_total)
X_noisy = X_noisy[perm]
X_clean = X_clean[perm]   

train_size = int(N_total * TRAIN_RATIO)
train_noisy = X_noisy[:train_size]
valid_noisy = X_noisy[train_size:]
valid_clean = X_clean[train_size:]   

# ==================== Step 5: Create DataLoader . ====================
train_dataset = TensorDataset(train_noisy)   
valid_dataset = TensorDataset(valid_noisy)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Data loading complete.")

Loading AmirSyn2d and coh2d ...
Amir: noisy (19824, 1225), clean (19824, 1225)
coh:  noisy (3025, 1225), no clean data (用 noisy 顶替)
SWAN_real_poststack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz: 117351 blocks loaded
SWAN_real_prestack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz: 6264 blocks loaded
SWAN_syn_poststack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz: 67077 blocks loaded
SWAN_syn_prestack_10percent_high_quality_SNR-4dB_impulse10pct_random.npz: 292950 blocks loaded

Total noisy shape: (506491, 1225)
Total clean shape (for evaluation only): (506491, 1225)
Data loading complete.


In [7]:
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU(0.1, inplace=True)
        self.norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

In [8]:
class SparseAttention(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=2):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        block_size = self.window_size * 2
        num_blocks = max(1, (N + block_size - 1) // block_size)
        mask = torch.zeros(N, N, device=x.device)
        for i in range(num_blocks):
            start = i * block_size
            end = min(start + block_size, N)
            mask[start:end, start:end] = 1
        mask = mask.unsqueeze(0).unsqueeze(0).bool()

        attn = attn.masked_fill(~mask, -1e9)
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class Sparse_PatchAE(nn.Module):
    def __init__(self, input_dim, embed_dim=256, depth=1, num_heads=4, window_size=2, num_tokens=4):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        self.num_tokens = num_tokens
        # 每个 token 的维度 = embed_dim / num_tokens
        self.token_dim = embed_dim // num_tokens

        # 嵌入层：input_dim -> embed_dim（和原代码一致）
        self.embed = nn.Linear(input_dim, embed_dim)

        self.cls_token = nn.Parameter(torch.randn(1, 1, self.token_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens + 1, self.token_dim) * 0.02)

        self.num_heads = self._pick_num_heads(num_heads, self.token_dim)
        self.window_size = window_size

        self.blocks = nn.ModuleList([
            self._make_block(self.token_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(self.token_dim)
        self.decoder = nn.Linear(embed_dim, input_dim)

    def _pick_num_heads(self, requested, token_dim):
        """Return the largest heads <= requested that evenly divides token_dim."""
        for h in range(min(requested, token_dim), 0, -1):
            if token_dim % h == 0:
                return h
        return 1

    def _make_block(self, dim):
        return nn.ModuleList([
            nn.LayerNorm(dim),
            SparseAttention(dim, self.num_heads, self.window_size),
            nn.LayerNorm(dim),
            nn.Sequential(
                nn.Linear(dim, dim * 4),
                nn.GELU(),
                nn.Linear(dim * 4, dim)
            )
        ])

    def forward(self, x):
        B = x.shape[0]

        
        x = self.embed(x)                       

        x = x.reshape(B, self.num_tokens, self.token_dim)   

        cls_tokens = self.cls_token.expand(B, -1, -1)       
        x = torch.cat((cls_tokens, x), dim=1)               
        x = x + self.pos_embed                              

        for norm1, attn, norm2, mlp in self.blocks:
            x = x + attn(norm1(x))
            x = x + mlp(norm2(x))

        x = self.norm(x)
        patch_tokens = x[:, 1:, :]                          
        patch_tokens = patch_tokens.reshape(B, -1)          

        recon = self.decoder(patch_tokens)                  
        return recon

In [9]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()
        self.fcb1 = FCB(input_size, 512, dropout)
        self.Sparse1 = Sparse_PatchAE(512, 512, depth=4)
        
        self.fcb2 = FCB(512, 128, dropout)
        self.Sparse2 = Sparse_PatchAE(128, 128, depth=4)
        
        self.fcb3 = FCB(128, 64, dropout)
        self.Sparse3 = Sparse_PatchAE(64, 64, depth=4)
        
        self.fcb4 = FCB(64, 32, dropout)
        self.Sparse4 = Sparse_PatchAE(32, 32, depth=4)
        
        self.fcb5 = FCB(32, 8, dropout)
        self.Sparse5 = Sparse_PatchAE(8, 8, depth=4)

    def forward(self, x):
        x1 = self.fcb1(x)
        y1 = self.Sparse1(x1)
        x2 = self.fcb2(x1)
        y2 = self.Sparse2(x2)
        x3 = self.fcb3(x2)
        y3 = self.Sparse3(x3)
        x4 = self.fcb4(x3)
        y4 = self.Sparse4(x4)
        x5 = self.fcb5(x4)
        y5 = self.Sparse5(x5)
        return x1, x2, x3, x4, x5, y1, y2, y3, y4, y5

In [10]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.fcb5 = FCB(8 + 8, 32, dropout)
        self.fcb4 = FCB(32 + 32, 64, dropout)
        self.fcb3 = FCB(64 + 64, 128, dropout)
        self.fcb2 = FCB(128 + 128, 512, dropout)
        self.fcb1 = FCB(512 + 512, output_size, dropout)

    def forward(self, x1, x2, x3, x4, x5, y1, y2, y3, y4, y5):
        y51 = torch.cat([x5, y5], dim=1)
        y41 = self.fcb5(y51)
        y41 = torch.cat([y41, y4], dim=1)
        y31 = self.fcb4(y41)
        y31 = torch.cat([y31, y3], dim=1)
        y21 = self.fcb3(y31)
        y21 = torch.cat([y21, y2], dim=1)
        y11 = self.fcb2(y21)
        y11 = torch.cat([y11, y1], dim=1)
        output = self.fcb1(y11)
        return output

In [11]:
class AutoCoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_size, dropout)
        self.decoder = Decoder(output_size, dropout)

    def forward(self, x):
        x1, x2, x3, x4, x5, y1, y2, y3, y4, y5 = self.encoder(x)
        output = self.decoder(x1, x2, x3, x4, x5, y1, y2, y3, y4, y5)
        return output

In [12]:
input_dim=w1*w2
model = AutoCoder(input_size=input_dim, output_size=input_dim)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 3,546,787


In [13]:
def huber_loss(pred, target, delta=1.35):
    error = pred - target
    abs_error = torch.abs(error)
    quadratic = torch.clamp(abs_error, max=delta)
    linear = abs_error - quadratic
    loss = 0.5 * quadratic ** 2 + delta * linear
    return loss.mean()

In [14]:
import torch
import torch.nn as nn
import time
from torch.nn.functional import huber_loss

w1, w2 = 35, 35
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


use_validation = len(valid_noisy) > 0
if not use_validation:
    print("(valid_loss use train_loss ）")

# ==================== Stage 1: Model initialization + train for the first 15 epochs ====================
print("Start Stage 1 training: first 15 epochs.")
model = AutoCoder(input_size=w1*w2, output_size=w1*w2, dropout=0.1).to(device)
model.decoder.fcb1 = nn.Linear(1024, w1*w2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
start_time = time.time()

es_thres = 5
es_cnt = 0
prev_valid_loss = float('inf')
loss_train = []
loss_valid = []
num_epochs_stage1 = 15
mask_ratio1 = 0.3
mask_ratio2 = 0.1

# ---------------- Training loop ----------------
for epoch in range(num_epochs_stage1):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        noisy_patch = batch[0].to(device)
        B = noisy_patch.shape[0]

        rand_tensor = torch.rand_like(noisy_patch)
        random_mask_2d = rand_tensor < mask_ratio1
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch

        optimizer.zero_grad()
        outputs = model(input_patch)
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.35)

        loss = rec_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * input_patch.size(0)

    train_loss = train_loss / len(train_noisy)
    loss_train.append(train_loss)

    # ---------------- Validation phase ----------------
    val_start = time.time()
    if use_validation:
        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for batch in valid_loader:
                valid_patch = batch[0].to(device)

                rand_tensor_val = torch.rand_like(valid_patch)
                random_mask_2d_val = rand_tensor_val < mask_ratio2
                input_val_patch = valid_patch * (~random_mask_2d_val)

                outputs = model(input_val_patch)
                rec_loss_val = huber_loss(outputs[random_mask_2d_val],
                                          valid_patch[random_mask_2d_val],
                                          delta=1.35)
                valid_loss += rec_loss_val.item() * valid_patch.size(0)

        valid_loss = valid_loss / len(valid_noisy)
    else:
        
        valid_loss = train_loss

    loss_valid.append(valid_loss)
    val_time = time.time() - val_start
    epoch_time = time.time() - epoch_start

    print(f"stage1 - Epoch[{epoch+1}/{num_epochs_stage1}], "
          f"Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, "
          f"Epoch Time: {epoch_time:.2f}s, Val Time: {val_time:.2f}s")

    # Saving best model
    if valid_loss < prev_valid_loss:
        prev_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model_35,35,15+15_stage1.pth')
        print("The optimal model of Stage 1 has been saved.")

# ==================== Stage 2: Load optimal parameters + continue training for another 15 epochs ====================
print("\nStart Stage 2 training: load the optimal model and train for another 15 epochs.")
prev_valid_loss = float('inf')
model.load_state_dict(torch.load('best_model_35,35,15+15_stage1.pth'))
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs_stage2 = 15

# ---------------- Training loop ----------------
for epoch in range(num_epochs_stage2):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        noisy_patch = batch[0].to(device)
        B = noisy_patch.shape[0]

        rand_tensor = torch.rand_like(noisy_patch)
        random_mask_2d = rand_tensor < mask_ratio1
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch

        optimizer.zero_grad()
        outputs = model(input_patch)
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.35)

        loss = rec_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * input_patch.size(0)

    train_loss = train_loss / len(train_noisy)
    loss_train.append(train_loss)

    # ---------------- Validation phase ----------------
    val_start = time.time()
    if use_validation:
        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for batch in valid_loader:
                valid_patch = batch[0].to(device)

                rand_tensor_val = torch.rand_like(valid_patch)
                random_mask_2d_val = rand_tensor_val < mask_ratio2
                input_val_patch = valid_patch * (~random_mask_2d_val)

                outputs = model(input_val_patch)
                rec_loss_val = huber_loss(outputs[random_mask_2d_val],
                                          valid_patch[random_mask_2d_val],
                                          delta=1.35)
                valid_loss += rec_loss_val.item() * valid_patch.size(0)

        valid_loss = valid_loss / len(valid_noisy)
    else:
        valid_loss = train_loss

    loss_valid.append(valid_loss)
    val_time = time.time() - val_start
    epoch_time = time.time() - epoch_start

    print(f"stage2 - Epoch[{epoch+1}/{num_epochs_stage2}], "
          f"Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, "
          f"Epoch Time: {epoch_time:.2f}s, Val Time: {val_time:.2f}s")

    # Saving best model
    if valid_loss < prev_valid_loss:
        prev_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model_35,35,15+15_final.pth')
        print("The optimal model of Stage 2 has been saved")

total_time = time.time() - start_time
print("\n" + "="*50)
print("All training procedures completed!")
print(f"Total training time: {total_time/60:.2f} minutes ({total_time:.2f} seconds)")
print(f"Final best model saved as 'best_model_35,35,15+15_final.pth'")


if torch.cuda.is_available():
    max_mem = torch.cuda.max_memory_allocated() / 1024**3
    current_mem = torch.cuda.memory_allocated() / 1024**3
    print(f"Peak GPU memory allocated: {max_mem:.2f} GB")
    print(f"Current GPU memory allocated: {current_mem:.2f} GB")
else:
    print("CUDA not available, no GPU memory statistics.")
print("="*50)

⚠️  验证集为空，训练过程将跳过验证阶段（valid_loss 使用 train_loss 代替）。
Start Stage 1 training: first 15 epochs.
stage1 - Epoch[1/15], Train Loss: 0.1116, Valid Loss: 0.1116, Epoch Time: 44.22s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[2/15], Train Loss: 0.0973, Valid Loss: 0.0973, Epoch Time: 43.70s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[3/15], Train Loss: 0.0956, Valid Loss: 0.0956, Epoch Time: 45.53s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[4/15], Train Loss: 0.0947, Valid Loss: 0.0947, Epoch Time: 46.17s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[5/15], Train Loss: 0.0940, Valid Loss: 0.0940, Epoch Time: 45.58s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[6/15], Train Loss: 0.0935, Valid Loss: 0.0935, Epoch Time: 45.43s, Val Time: 0.00s
The optimal model of Stage 1 has been saved.
stage1 - Epoch[7/15], Train Loss: 0.0932, Valid L